# TPU v5e (Colab) + GPU (PSC L40S): routing experiments

The router runs here, next to vLLM-TPU. The GPU server runs on PSC; we reach it through an SSH
tunnel via the PSC login node (Colab can connect out, nothing can connect in).

```
Colab (TPU v5e)
 ├─ vLLM-TPU            127.0.0.1:8000   "tpu"
 ├─ ssh tunnel          127.0.0.1:8001 ──► bridges2.psc.edu ──► <PSC node>:8000   "gpu"
 └─ replay/replay.py → router/router.py → one of the two
```

**Before running:**
1. Runtime → Change runtime type → **TPU v5e**.
2. On PSC, start the GPU server with the **same `MODEL` and `MAX_LEN`** as cell 2, either
   `sbatch --export=ALL,MODEL=Qwen/Qwen3-4B,MAX_LEN=16384 scripts/psc/serve.sbatch` (then wait for
   `READY node=...` in `slurm-har-serve-<jobid>.out`), or inside `interact`:
   `HOST=0.0.0.0 MODEL=Qwen/Qwen3-4B MAX_LEN=16384 bash scripts/psc/serve.sh 8000 0.85` (node: `hostname -s`).

Cells 1–7 set things up and check them. Cells 8–12 are the experiments.

In [ ]:
# 1. Repo + vLLM-TPU (skips the install if vllm already imports)
BRANCH = "main"   # or a branch you're testing, e.g. "yanze/colab-tpu-psc"
%cd /content
!test -d hetero-agent-routing || git clone -q https://github.com/clemrho/hetero-agent-routing
%cd /content/hetero-agent-routing
!git fetch -q && git checkout -q {BRANCH} && git pull -q
!git log --oneline -1
# vllm-tpu has no wheel for Colab's Python 3.13, so it gets its own Python 3.12 venv
# (via uv). Everything else in this notebook uses Colab's Python.
VENV = "/content/vllm_env"
!pip install -q uv httpx
!test -x {VENV}/bin/python || uv venv -q --python 3.12 {VENV}
!{VENV}/bin/python -c "import vllm" 2>/dev/null || uv pip install -q --python {VENV}/bin/python vllm-tpu
!{VENV}/bin/python -c "import jax, vllm; print('vllm', vllm.__version__); print(jax.devices())"

In [ ]:
# 2. Start vLLM-TPU. start_tpu() restarts the server if it runs a different model or length.
#    The first start of a given length compiles with XLA: can take 10+ min.
import json, os, subprocess, time, urllib.request

REPO = "/content/hetero-agent-routing"
MODEL = "Qwen/Qwen3-4B"   # must match the PSC server
MAX_LEN = 16384           # must match the PSC server
os.makedirs(f"{REPO}/logs", exist_ok=True)

def served(port):
    """(model, max_model_len) served on a port, or None if nothing answers."""
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=5) as r:
            d = json.load(r)["data"][0]
        return d["id"], d.get("max_model_len")
    except Exception:
        return None

def up(port):
    return served(port) is not None

TPU_PROC = None

def start_tpu(model, max_len):
    global TPU_PROC
    cur = served(8000)
    if cur and cur[0] == model and cur[1] in (max_len, None):
        print(f"vLLM-TPU already running {cur}")
        return
    if cur:
        print(f"stopping vLLM-TPU running {cur}")
        if TPU_PROC is not None:
            TPU_PROC.terminate()
        subprocess.run(["pkill", "-f", "vllm serve"])
        for _ in range(60):
            if not up(8000):
                break
            time.sleep(2)
        else:
            raise RuntimeError("old vLLM-TPU did not stop; restart the Colab runtime")
        time.sleep(10)   # let the old process release the TPU
    log_path = f"{REPO}/logs/vllm_tpu_{max_len}.log"
    TPU_PROC = proc = subprocess.Popen(
        [f"{VENV}/bin/vllm", "serve", model, "--port", "8000", "--max-model-len", str(max_len),
         "--enable-prefix-caching", "--enable-prompt-tokens-details"],
        stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    t0 = time.time()
    while not up(8000):
        if proc.poll() is not None:
            print(open(log_path).read()[-4000:])
            raise RuntimeError("vLLM-TPU exited, see the log above")
        print(f"waiting for vLLM-TPU ({model}, max_len={max_len})... {time.time() - t0:.0f}s", end="\r")
        time.sleep(10)
    print(f"\nvLLM-TPU ready on :8000 {served(8000)}")

start_tpu(MODEL, MAX_LEN)

In [ ]:
# 3. Check the TPU server: cold vs warm request, cached_tokens > 0 on warm
!python {REPO}/scripts/check_backend.py --url http://127.0.0.1:8000 --model {MODEL}

In [58]:
# 4. SSH tunnel to the PSC GPU: 127.0.0.1:8001 -> <PSC_NODE>:<PSC_PORT>
PSC_USER = "ylu36"
PSC_NODE = "gl003"     # READY line of slurm-har-serve-<jobid>.out, or `hostname -s` in interact
PSC_PORT = 8000

import getpass
!apt-get -qq install -y sshpass > /dev/null
!pkill -f "ssh -f -N -o StrictHostKeyChecking" || true
os.environ["SSHPASS"] = getpass.getpass(f"PSC password for {PSC_USER}: ")
r = subprocess.run(
    f"sshpass -e ssh -f -N -o StrictHostKeyChecking=accept-new -o ServerAliveInterval=30 "
    f"-o ExitOnForwardFailure=yes -L 8001:{PSC_NODE}:{PSC_PORT} {PSC_USER}@bridges2.psc.edu",
    shell=True)
del os.environ["SSHPASS"]
for _ in range(10):
    if up(8001):
        break
    time.sleep(2)
print(f"tunnel up: GPU serves {served(8001)}" if up(8001) else
      "!! tunnel failed: check user/password/node, and that the PSC server is still running")

^C
PSC password for ylu36: ··········
tunnel up: GPU serves ('Qwen/Qwen3-4B', 32768)


In [59]:
# 5. Check the GPU server through the tunnel
!python {REPO}/scripts/check_backend.py --url http://127.0.0.1:8001 --model {MODEL}

http://127.0.0.1:8001 serves ['Qwen/Qwen3-4B']
  cold ok (412.6 ms), sending warm request (prefix-cache hit)...
  cold:   412.6 ms  prompt_tokens=3033  cached_tokens=0
  warm:   226.7 ms  prompt_tokens=3033  cached_tokens=3024
OK  speedup cold/warm = 1.8x


In [ ]:
# 6. Router smoke test with synthetic workflows (LB, 4 workflows): checks the whole path quickly
run_dir = f"{REPO}/logs/colab_{int(time.time())}"   # fresh dir, so old runs don't mix in
os.makedirs(run_dir)
os.environ.update(TPU_URL="http://127.0.0.1:8000", GPU_URL="http://127.0.0.1:8001", MODEL=MODEL)
!cd {run_dir} && python {REPO}/router/router.py --policy LB --n_wf 4
!python {REPO}/analysis/analyze.py {run_dir}/logs/*.jsonl

In [ ]:
# 7. Network round trip to each backend (the GPU one goes Colab -> PSC; report it with the results)
import statistics
for port, name in [(8000, "tpu (local)"), (8001, "gpu (via PSC tunnel)")]:
    ts = []
    for _ in range(10):
        t = time.time()
        urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=10).read()
        ts.append((time.time() - t) * 1000)
    print(f"{name}: median RTT {statistics.median(ts):.1f} ms")

## Experiments

Each experiment generates its own trace, replays it for every policy × concurrency, prints the
summary, and saves everything under `logs/<name>/` (plus Google Drive if cell 8 mounted it).
Re-running an experiment skips the runs that already finished, so a disconnect only costs the
run in progress: reconnect, re-run cells 1, 2, 4, 8 and the experiment cell.

In [60]:
# 8. Helpers: Drive for saving logs, and run_experiment()
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/har"

def sh(cmd):
    get_ipython().system(cmd)

def run_experiment(name, gen_args, conc, policies="LB FF CA RR", reps=1, limit=None):
    # both servers must serve the same model and max_model_len as this notebook
    for port, dev in [(8000, "tpu"), (8001, "gpu")]:
        s = served(port)
        if s is None:
            raise RuntimeError(f"{dev} server on :{port} is not answering")
        if s[0] != MODEL or (s[1] is not None and s[1] != MAX_LEN):
            raise RuntimeError(f"{dev} serves {s}, but this notebook expects ({MODEL!r}, {MAX_LEN})")
    trace, log_dir = f"{REPO}/logs/{name}.jsonl", f"{REPO}/logs/{name}"
    warm = " ".join(str(n) for n in (1024, 2048, 4096, 8192, 16384, 24576) if n < MAX_LEN)
    os.environ.update(TPU_URL="http://127.0.0.1:8000", GPU_URL="http://127.0.0.1:8001", MODEL=MODEL)
    sh(f"python {REPO}/traces/gen_synthetic.py --max_model_len {MAX_LEN} --out {trace} {gen_args}")
    sh(f"python {REPO}/replay/replay.py --trace {trace} --sweep --policies {policies} "
       f"--concurrency_list {conc} --reps {reps} {f'--limit {limit}' if limit else ''} "
       f"--warmup {warm} --log_dir {log_dir}")
    sh(f"python {REPO}/analysis/analyze.py {log_dir}/*/*_r*.jsonl | tee {log_dir}/summary.txt")
    sh(f"cp {trace} {log_dir}/")
    if os.path.isdir("/content/drive/MyDrive"):
        sh(f"mkdir -p {DRIVE_DIR} && cp -r {log_dir} {DRIVE_DIR}/ && echo saved to {DRIVE_DIR}/{name}")

# Trace shapes. Short group: contexts 2k -> ~10k. Long group: 8k-16k -> up to ~27k.
SHORT = "--steps 3 5 --init_ctx 2048 {ctx} --out_min 20 --out_max 60 --tool_min 0.2 --tool_max 0.8"
LONG = ("--steps 3 5 --init_ctx 8192 16384 --obs_min 1500 --obs_max 3000 "
        "--out_min 20 --out_max 60 --tool_min 0.5 --tool_max 2.0")
print("helpers ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
helpers ready


In [ ]:
# 9. Quick baseline (~5 min): 12 workflows, concurrency 4 and 8. Needs MAX_LEN = 16384.
run_experiment("quick12", SHORT.format(ctx=MAX_LEN // 2) + " --per_combo 3", conc="4 8")

In [ ]:
# 10. Short context, high load (~15-30 min): 64 workflows so that 32 really run at once.
#     Expect the TPU (16 GB) to run out of KV-cache space at 32: hit rates drop even for CA.
run_experiment("short64_c8-32", SHORT.format(ctx=MAX_LEN // 2) + " --per_combo 16", conc="8 16 32")

### Long-context group (contexts up to ~27k)

Needs both servers at `MAX_LEN = 32768`:
1. **PSC**: stop the GPU server (`scancel <jobid>`, or Ctrl+C in `interact`) and start it again with
   `MAX_LEN=32768`, e.g. `sbatch --export=ALL,MODEL=Qwen/Qwen3-4B,MAX_LEN=32768 scripts/psc/serve.sbatch`.
   If it lands on a different node, re-run cell 4 with the new `PSC_NODE`.
2. Run cell 11 (restarts vLLM-TPU at 32768; XLA compiles again), then cell 12.

A 32k request needs ~4.7 GB of KV cache; the v5e has ~8 GB left after the 4B weights, so keep
concurrency low. If vLLM-TPU fails to start or errors at 32768, use 24576 on both sides.

In [57]:
# 11. Switch both sides to MAX_LEN = 32768 (restart the PSC server first, see above)
MAX_LEN = 32768
start_tpu(MODEL, MAX_LEN)
print("tpu:", served(8000), " gpu:", served(8001))


vLLM-TPU ready on :8000 ('Qwen/Qwen3-4B', 32768)
tpu: ('Qwen/Qwen3-4B', 32768)  gpu: None


In [ ]:
# 12. Long context (~30-60 min): 16 workflows, 8k-16k start, ~2k tokens added per step,
#     0.5-2 s tool waits, concurrency 2 / 4 / 8.
run_experiment("long16_c2-8", LONG + " --per_combo 4", conc="2 4 8")

wrote 16 workflows -> /content/hetero-agent-routing/logs/long16_c2-8.jsonl   (max_model_len=32768)
steps  ctx0   n kept steps  max ctx truncated
    3  8192   4        [3]    14011      0/4
    3 16384   4        [3]    21744      0/4
    5  8192   4        [5]    18354      0/4
    5 16384   4        [5]    27124      0/4
total LLM calls=64, total tool wait=62s (lower bound on a concurrency-1 run, before any LLM time)
trace /content/hetero-agent-routing/logs/long16_c2-8.jsonl: 16 workflows, 64 LLM calls; backends {'a100': 'http://localhost:8001', 'tpu': 'http://127.0.0.1:8000'}
  warmup a100 (http://localhost:8001): 6.7s for lengths [1024, 2048, 4096, 8192, 16384, 24576]
  warmup  tpu (http://127.0.0.1:8000): 6.0s for lengths [1024, 2048, 4096, 8192, 16384, 24576]
skip /content/hetero-agent-routing/logs/long16_c2-8/long16_c2-8/LB_c2_r0.jsonl (exists; --overwrite to redo)
    FF_c2_r0: 4/16 done, 0 failed, 30s
    FF_c2_r0: 11/16 done, 0 failed, 60s
FF_c2_r0: 16/16 workflows in 80.6s -

**Reading the results:** `devices` shows how many calls went to each device; `cache_hit` is
compared with its upper bound `(max ...)`. TTFT on `gpu` includes the network round trip from
cell 7. Copy `logs/<name>/` (or the Drive copy) into `results/` in the repo for runs we report.

When done: stop the PSC server (`scancel <jobid>` or Ctrl+C), and run the cell below to close the tunnel.

In [ ]:
!pkill -f "ssh -f -N -o StrictHostKeyChecking" && echo "tunnel closed" || echo "no tunnel running"